In [2]:
from pathlib import Path
import json
import os
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import urlopen

from dotenv import load_dotenv

current_path = Path.cwd()
project_root = current_path if (current_path / "notebooks").exists() else current_path.parent

notebooks_dir = project_root / "notebooks"
env_example_path = project_root / ".env.example"
env_path = project_root / ".env"
gitignore_path = project_root / ".gitignore"
previous_notebook_path = notebooks_dir / "04_basic_charts.ipynb"
previous_chart_path = project_root / "assets" / "basic_chart.png"

print("project_root:", project_root)
print("notebooks_dir_exists:", notebooks_dir.exists())
print("previous_notebook_exists:", previous_notebook_path.exists())
print("previous_chart_exists:", previous_chart_path.exists())
print("env_example_exists:", env_example_path.exists())
print("env_file_exists:", env_path.exists())
print("gitignore_exists:", gitignore_path.exists())

project_root: /Users/im-youngchan/Desktop/US Financial
notebooks_dir_exists: True
previous_notebook_exists: True
previous_chart_exists: True
env_example_exists: True
env_file_exists: True
gitignore_exists: True


In [7]:
sample_response_text = """
{
    "count": 2, 
    "observations":[
        {
            "date": "2023-01-01",
            "value": "3.4"
        },
        {
            "date": "2023-02-01",
            "value": "3.6"
        }
    ]
}
"""

sample_response_json = json.loads(sample_response_text)
sample_observations = sample_response_json["observations"]
sample_first_observation = sample_observations[0]

print("json_pyhton_type:", type(sample_response_json).__name__)
print("top_level_keys:", list(sample_response_json.keys()))
print("observation_count:", len(sample_observations))
print("first_observation:", sample_first_observation)
print("first_value_python_type:", type(sample_first_observation["value"]).__name__)

json_pyhton_type: dict
top_level_keys: ['count', 'observations']
observation_count: 2
first_observation: {'date': '2023-01-01', 'value': '3.4'}
first_value_python_type: str


In [8]:
load_dotenv(dotenv_path=env_path, override=False)
fred_api_key = os.getenv("FRED_API_KEY")

invalid_key_values = {
    None, 
    "",
    "replace_with_your_fred_api_key",
}

if fred_api_key in invalid_key_values:
    raise ValueError(
        "FRED_API_KEY가 준비되지 않았습니다. "
        "프로젝트 루트의 .env에 실제 key를 저장한 뒤 다시 실행하세요."
    )

print("fred_api_key_loaded:", True)

fred_api_key_loaded: True


In [9]:
fred_endpoint = "https://api.stlouisfed.org/fred/series/observations"
series_id = "UNRATE"
observation_start = "2023-01-01"
observation_end = "2023-12-31"
observation_limit = 5
sort_order = "asc"

request_params = {
    "series_id": series_id,
    "api_key": fred_api_key, 
    "file_type": "json",
    "observation_start": observation_start,
    "observation_end": observation_end, 
    "limit": observation_limit,
    "sort_order": sort_order, 
}

safe_request_params = {
    **request_params,
    "api_key": "***",
}

print("fred_endpoint:", fred_endpoint)
print("safe_request_params:", safe_request_params)

fred_endpoint: https://api.stlouisfed.org/fred/series/observations
safe_request_params: {'series_id': 'UNRATE', 'api_key': '***', 'file_type': 'json', 'observation_start': '2023-01-01', 'observation_end': '2023-12-31', 'limit': 5, 'sort_order': 'asc'}


In [10]:
request_query = urlencode(request_params)
request_url_with_key = f"{fred_endpoint}?{request_query}"

try:
    with urlopen(request_url_with_key, timeout=30) as response:
        response_status = response.status
        response_content_type = response.headers.get_content_type()
        response_text = response.read().decode("utf-8")
except HTTPError as error:
    raise RuntimeError(
        f"FRED API request 가 HTTP 상태 {error.code}로 실패했습니다."
    ) from error
except URLError as error:
    raise RuntimeError(
        f"FRED API 연결에 실패했습니다: {error.reason}"
    ) from error

print("response_status:", response_status)
print("response_content_type:", response_content_type)
print("response_text_length:", len(response_text))


response_status: 200
response_content_type: application/json
response_text_length: 740


In [11]:
fred_response_json = json.loads(response_text)
observations = fred_response_json.get("observations", [])

print("response_python_type:", type(fred_response_json).__name__)
print("response_top_level_keys:", list(fred_response_json.keys()))
print("response_count:", fred_response_json.get("count"))
print("returned_observation_count:", len(observations))
print("returned_limit_match:", len(observations) == observation_limit)


response_python_type: dict
response_top_level_keys: ['realtime_start', 'realtime_end', 'observation_start', 'observation_end', 'units', 'output_type', 'file_type', 'order_by', 'sort_order', 'count', 'offset', 'limit', 'observations']
response_count: 12
returned_observation_count: 5
returned_limit_match: True


In [12]:
if not observations:
    raise ValueError("observations 목록이 비어 있습니다. request parameter를 확인하세요.")

first_observation = observations[0]
required_observation_keys = {"date", "value"}

print("first_observation_keys:", list(first_observation.keys()))
print("observation_keys_ready:", required_observation_keys.issubset(first_observation))
print("first_observation_date:", first_observation["date"])
print("first_observation_value_text:", first_observation["value"])
print("first_value_python_type:", type(first_observation["value"]).__name__)
print(json.dumps(first_observation, indent=2))

first_observation_keys: ['realtime_start', 'realtime_end', 'date', 'value']
observation_keys_ready: True
first_observation_date: 2023-01-01
first_observation_value_text: 3.5
first_value_python_type: str
{
  "realtime_start": "2026-07-21",
  "realtime_end": "2026-07-21",
  "date": "2023-01-01",
  "value": "3.5"
}


In [13]:
safe_request_text = json.dumps(safe_request_params, sort_keys=True)

print("safe_request_has_mask:", '"api_key:" "***"' in safe_request_text)
print("actual_key_hidden_in_preview:", fred_api_key not in safe_request_text)

safe_request_has_mask: False
actual_key_hidden_in_preview: True
